In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

#sklearn classification models 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier


#Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder

#Dimensionality Reduction
from sklearn.decomposition import PCA

#Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

#Evaluation Metrics F1, ROC OVR
# Balanced Accuracy,

# · The macro-average and micro-average of the One-vs-Rest ROC AUC score.

# In addition,

# · Draw the macro-average One-vs-Rest ROCs for all classification methods into a single graph to allow for easy comparison between methods.

# · Draw the micro-average One-vs-Rest ROCs for all classification methods into a single graph to allow for easy comparison between methods.

# · For each class, draw the One-vs-Rest ROCs for all classification methods into a separate graph, to allow for easy comparison between methods.
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

#Confusion Matrix
from sklearn.metrics import confusion_matrix





In [2]:
#data load
from sklearn.datasets import fetch_openml

dataset = fetch_openml(data_id=4538, as_frame=False) # fetch the dataset


In [3]:

X = dataset.data
y = dataset.target


print(X.shape)
print(y.shape)

#checking integrity of data
distribution = np.unique(y, return_counts=True)
isnan = np.isnan(X).sum()

print("general distribution of data :", distribution)
print("Nans in data :", isnan)


(9873, 32)
(9873,)
general distribution of data : (array(['D', 'H', 'P', 'R', 'S'], dtype=object), array([2741,  998, 2097, 1087, 2950]))
Nans in data : 0


In [4]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

outlier_labels = iso.fit_predict(X)
# -1 = outlier, 1 = inlier

n_outliers = (outlier_labels == -1).sum()
print("Detected outliers:", n_outliers)

Detected outliers: 494


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, stratify=y, random_state=42)
#for KNN might delete later
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)


In [15]:
#Model Selection without hyperparameter tuning

models = [
    LogisticRegression(),
    DecisionTreeClassifier(),
    RandomForestClassifier(),
    KNeighborsClassifier(),
    LinearSVC(),
    SVC(),
    GaussianNB()
]

model_names = [
    "Logistic Regression",
    "Decision Tree",
    "Random Forest",
    "KNN",
    "Linear SVC",
    "SVC",
    "Naive Bayes"
]

results = {}
for name, model in zip(model_names, models):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = balanced_accuracy_score(y_test, y_pred)
    results[name] = acc

print(results)

{'Logistic Regression': 0.30748896861726255, 'Decision Tree': 0.468910998587328, 'Random Forest': 0.5870761006573666, 'KNN': 0.5581649197515796, 'Linear SVC': 0.3062805205741361, 'SVC': 0.4300664411211203, 'Naive Bayes': 0.3425543766949714}


In [ ]:
#hyperparameter tuning with and finding out the best hyper parameter
models = {
     "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),
    
    "Logistic Regression (PCA)": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("pca", PCA()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5, 10, 15, 20, 25],
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),

    "Decision Tree": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 5, 10]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "n_estimators": [200, 500],
            "max_depth": [None, 15, 30],
            "min_samples_leaf": [1, 5]
        }
    ),   

    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                algorithm="brute",
                n_jobs=1        # REQUIRED
            ))
        ]),
        {
            "clf__n_neighbors": list(range(1, 51, 2)),
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"]
        }
    ),

    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                class_weight="balanced",
                max_iter=5000
            ))
        ]),
        {
            "clf__C": [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100], 
            "clf__loss": ["squared_hinge"],
            "clf__dual": [False] 
        }
    ),
    
    "SVC (RBF)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False
            ))
        ]),
        {
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": [0.001, 0.01, 0.1]
        }
    ),

    "Naive Bayes": (
        GaussianNB(),
        {
            "var_smoothing": np.logspace(-12, -6, 20)
        }
    ),
    
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
            "subsample": [0.8, 1.0]
        }
    )
}



tuned_results = {}
best_models = {}

for name, (model, params) in models.items():
    print(f"\nTuning {name} ...")

    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results[name] = acc
    best_models[name] = grid.best_params_

print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models.items():
    print(f"{k}: {v}")



Tuning Logistic Regression ...

Tuning Logistic Regression (PCA) ...

Tuning Decision Tree ...

Tuning Random Forest ...

Tuning KNN ...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 409, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>.


Tuning Linear SVC ...

Tuning SVC (RBF) ...

Tuning Naive Bayes ...

Tuning Gradient Boosting ...


In [32]:
X_train.min() >= 0 and X_test.min() >= 0


np.False_

## Logistic Regression

In [ ]:
models_lr = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),
    "Logistic Regression (PCA)": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("pca", PCA()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5, 10, 15, 20, 25],
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),

}

tuned_results_lr = {}
best_models_lr = {}

for name, (model, params) in models_lr.items():    
    print(f"\nTuning {name} ...")

    search = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_lr[name] = acc
    best_models_lr[name] = grid.best_params_


print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results_lr.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models_lr.items():
    print(f"{k}: {v}")



Tuned Results (Balanced Accuracy):
Logistic Regression: 0.4363
Logistic Regression (PCA): 0.4191

Best Hyperparameters:
Logistic Regression: {'clf__C': 10, 'clf__gamma': 0.1}
Logistic Regression (PCA): {'clf__C': 10, 'clf__gamma': 0.1}


## Decision Tree

In [ ]:
models_dt = {
    "Decision Tree": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "max_depth": [None, 10, 20, 30],
            "min_samples_leaf": [1, 5, 10]
        }
    ),
    
    "Decision Tree with Hyperparameters": (
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "criterion": ["gini", "entropy", "log_loss"],
            "max_depth": [5, 10, 20, 40, None],
            "min_samples_split": [2, 5, 10, 20],
            "min_samples_leaf": [1, 2, 5, 10],
            "ccp_alpha": [0.0, 0.001, 0.01, 0.1]
        }
        
    ),
}

tuned_results_dt = {}
best_models_dt = {}

for name, (model, params) in models_dt.items():
    print(f"\nTuning {name} ...")

    search = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    
    best_model = search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_dt[name] = acc
    best_models_dt[name] = grid.best_params_

print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results_dt.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models_dt.items():
    print(f"{k}: {v}")


Tuned Results (Balanced Accuracy):
Decision Tree: 0.4729
Decision Tree with Hyperparameters: 0.4729

Best Hyperparameters:
Decision Tree: {'clf__C': 10, 'clf__gamma': 0.1}
Decision Tree with Hyperparameters: {'clf__C': 10, 'clf__gamma': 0.1}


## Random Forest

In [ ]:
models_rf = {
    "Random Forest": (
        RandomForestClassifier(
            class_weight="balanced",
            random_state=42
        ),
        {
            "n_estimators": [200, 500],
            "max_depth": [None, 15, 30],
            "min_samples_leaf": [1, 5]
        }
    ),   
}

tuned_results_rf = {}
best_models_rf = {}


for name, (model, params) in models_rf.items():    
    
    print(f"\nTuning {name} ...")
    
    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_rf[name] = acc
    best_models_rf[name] = grid.best_params_


print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results_rf.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models_rf.items():
    print(f"{k}: {v}")



Tuned Results (Balanced Accuracy):
Random Forest: 0.6052

Best Hyperparameters:
Random Forest: {'max_depth': None, 'min_samples_leaf': 5, 'n_estimators': 500}


In [ ]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

models_knn = {
    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                algorithm="brute",
                n_jobs=1        # REQUIRED
            ))
        ]),
        {
            "clf__n_neighbors": list(range(1, 51, 2)),
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"]
        }
    ),

    "KNN with minkowski": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                algorithm="brute",
                n_jobs=1        # REQUIRED
            ))
        ]),
        {
            "clf__n_neighbors": list(range(1, 51, 2)),
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["minkowski"],
            "clf__p": [1, 2]
        }
    )
}


tuned_results_knn = {}
best_models_knn = {}


for name, (model, params) in models_knn.items():
    print(f"\nTuning {name} ...")

    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1,
        error_score="raise"
    )
    
    grid.fit(X_train, y_train_enc)    
    
    best_model = grid.best_estimator_
    y_pred_enc = best_model.predict(X_test)
    y_pred = le.inverse_transform(y_pred_enc)

    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_knn[name] = acc
    best_models_knn[name] = grid.best_params_

print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results_knn.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models_knn.items():
    print(f"{k}: {v}")





Tuned Results (Balanced Accuracy):
KNN: 0.6354
KNN with minkowski: 0.6354

Best Hyperparameters:
KNN: {'clf__metric': 'manhattan', 'clf__n_neighbors': 1, 'clf__weights': 'uniform'}
KNN with minkowski: {'clf__metric': 'minkowski', 'clf__n_neighbors': 1, 'clf__p': 1, 'clf__weights': 'uniform'}


In [ ]:
models_svc = {
    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                class_weight="balanced",
                max_iter=5000
            ))
        ]),
        {
            "clf__C": [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100], 
            "clf__loss": ["squared_hinge"],
            "clf__dual": [False] 
        }
    ),
     "SVC (RBF)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False
            ))
        ]),
        {
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": [0.001, 0.01, 0.1]
        }
    ),
}

tuned_results_svc = {}
best_models_svc = {}
for name, (model, param_grid) in models_svc.items():
    print(f"\nTuning {name} ...")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=3,             
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = balanced_accuracy_score(y_test, y_pred)

    tuned_results_svc[name] = acc
    best_models_svc[name] = grid.best_params_

print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results_svc.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models_svc.items():
    print(f"{k}: {v}")





Tuning Linear SVC ...
Fitting 3 folds for each of 7 candidates, totalling 21 fits

Tuning SVC (RBF) ...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Tuned Results (Balanced Accuracy):
Linear SVC: 0.4085
SVC (RBF): 0.5166

Best Hyperparameters:
Linear SVC: {'clf__C': 10, 'clf__dual': False, 'clf__loss': 'squared_hinge'}
SVC (RBF): {'clf__C': 10, 'clf__gamma': 0.1}


In [ ]:

models_nb = {
    "Naive Bayes": (
        GaussianNB(),
        {
            "var_smoothing": np.logspace(-12, -6, 20)
        }
    )
}

tuned_results_nb = {}
best_models_nb = {}


for name, (model, params) in models_nb.items():
    print(f"\nTuning {name} ...")

    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results_nb[name] = acc
    best_models_nb[name] = grid.best_params_


print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results_nb.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models_nb.items():
    print(f"{k}: {v}")



Tuned Results (Balanced Accuracy):
Naive Bayes: 0.3426

Best Hyperparameters:
Naive Bayes: {'var_smoothing': np.float64(1e-12)}


In [ ]:
models_gb = {
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {
            "n_estimators": [100, 200],
            "learning_rate": [0.05, 0.1],
            "max_depth": [3, 5],
            "subsample": [0.8, 1.0]
        }
    )
}

tuned_results_gb = {}
best_models_gb = {}

for name, (model, param_grid) in models_gb.items():
    print(f"\nTuning {name} ...")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = balanced_accuracy_score(y_test, y_pred)

    tuned_results_gb[name] = acc
    best_models_gb[name] = grid.best_params_


print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results_gb.items():
    print(f"{k}: {v:.4f}")

print("\nBest Hyperparameters:")
for k, v in best_models_gb.items():
    print(f"{k}: {v}")


Tuning Gradient Boosting ...
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Tuned Results (Balanced Accuracy):
Gradient Boosting: 0.5951

Best Hyperparameters:
Gradient Boosting: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}


### Logistic Regression
{'Logistic Regression': 0.4349500231995328} with standard scalar
{'Logistic Regression': 0.43700762091970946} with robust scalar
{'Logistic Regression (PCA)': 0.41905043484826415} with robust scalar


### Grid Search `Decision`
{'Decision Tree': 0.472876417173258, 'Decision Tree with Hyperparameters': 0.472876417173258}

### Randomised Search CV `Decision`
{'Decision Tree': 0.472876417173258, 'Decision Tree with Hyperparameters': 0.472876417173258}

### Random Forest Randomised Search
{'Random Forest': 0.472876417173258}

### Random Forest Grid Search 
{'Random Forest': 0.6045172736609855}

### KNN Grid Search 
{'KNN': 0.5645810025619401}
### Tuned KNN
{'KNN': 0.6354202486328545, 'KNN with minkowski': 0.6354202486328545}



